# 03. Test Prediction and Submission

Notebook ini digunakan untuk menghasilkan prediksi pada data test menggunakan model EfficientNet-B0 terbaik dari proses training pada `02_modeling.ipynb`.

Model dimuat melalui checkpoint `b0_lr_high_best.pt`. Prediksi dilakukan menggunakan input gambar berukuran 224 × 224 piksel dan Test-Time Augmentation (TTA) dengan kombinasi resolusi 224 dan 256 piksel.

Hasil prediksi kemudian disimpan dalam format CSV sesuai ketentuan kompetisi Kaggle.

## 1. Setup

In [ ]:
from pathlib import Path
import sys
import zipfile
import numpy as np
import torch
from tqdm.auto import tqdm
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if Path("/kaggle/input").exists() and not (PROJECT_ROOT / "scripts" / "dataset.py").exists():
    script_zip = next(Path("/kaggle/input").rglob("scripts.zip"))
    PROJECT_ROOT = Path("/kaggle/working")
    with zipfile.ZipFile(script_zip) as z:
        z.extractall(PROJECT_ROOT)

sys.path.insert(0, str(PROJECT_ROOT))

from scripts.dataset import find_data_dir, load_metadata, find_image_root, make_loader
from scripts.evaluate import load_model
from scripts.predict import build_submission

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 16
NUM_WORKERS = 0
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else PROJECT_ROOT / "outputs"
SUBMISSION_DIR = OUTPUT_DIR if Path("/kaggle/working").exists() else PROJECT_ROOT / "submissions"
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

print("Device:", DEVICE)

## 2. Load Test Data

Data test dimuat menggunakan fungsi dari `scripts/dataset.py`. Metadata yang digunakan meliputi `test.csv`, `sample_submission.csv`, dan daftar kategori dokumen.

Dataset test berisi 7.994 gambar yang harus diklasifikasikan ke dalam 16 kategori dokumen.

Pada tahap ini, lokasi gambar test juga ditentukan untuk digunakan dalam proses inferensi. Data test tidak memiliki label yang diketahui, sehingga tidak digunakan untuk menghitung accuracy maupun loss.

In [ ]:
DATA_DIR = find_data_dir()
_, test_df, sample_submission, class_names, _ = load_metadata(DATA_DIR)
TEST_ROOT = find_image_root(DATA_DIR, test_df.iloc[0]["filename"])

print("Test images:", len(test_df))
print("Classes:", len(class_names))

## 3. Load Best Model

Model EfficientNet-B0 terbaik dimuat dari checkpoint b0_lr_high_best.pt yang dihasilkan pada notebook modeling.

Model kemudian diatur ke mode evaluasi menggunakan model.eval() agar proses inferensi berjalan tanpa memperbarui parameter model.

Checkpoint yang sama digunakan untuk prediksi original maupun prediksi menggunakan TTA.

In [ ]:
# Notebook 02 saves this file automatically after training.
CHECKPOINT_PATH = OUTPUT_DIR / "checkpoints" / "b0_lr_high_best.pt"

if not CHECKPOINT_PATH.is_file():
    local_path = PROJECT_ROOT / "checkpoints" / "b0_lr_high_best.pt"
    if local_path.is_file():
        CHECKPOINT_PATH = local_path
    elif Path("/kaggle/input").exists():
        matches = sorted(Path("/kaggle/input").rglob("b0_lr_high_best.pt"))
        if matches:
            CHECKPOINT_PATH = matches[0]

model, checkpoint = load_model(CHECKPOINT_PATH, DEVICE, expected_classes=class_names)
model.eval()

print("Checkpoint:", CHECKPOINT_PATH)
print("Best validation:", f"{checkpoint['best_accuracy']:.4%}")
print("Best epoch:", checkpoint.get("best_epoch", "not recorded"))

## 4. Test-Time Augmentation

Proses inferensi dilakukan menggunakan dua ukuran input gambar, yaitu 224 × 224 dan 256 × 256 piksel. Kedua ukuran menggunakan model dan checkpoint yang sama.

Setiap gambar menghasilkan probabilitas prediksi untuk 16 kelas dokumen. Probabilitas tersebut kemudian digabungkan menggunakan *weighted averaging* dengan bobot:

- 80% dari prediksi resolusi 224 piksel.
- 20% dari prediksi resolusi 256 piksel.

Label akhir ditentukan berdasarkan kelas dengan probabilitas gabungan tertinggi. TTA tidak mengubah bobot model karena hanya diterapkan selama proses inferensi.

In [ ]:
def predict_probabilities(image_size):
    """Predict class probabilities for each test image in its original CSV order."""
    loader = make_loader(
        test_df, TEST_ROOT, image_size=image_size,
        batch_size=BATCH_SIZE, training=False, labeled=False,
        num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda",)
    probabilities = []
    with torch.inference_mode():
        for images in tqdm(loader, desc=f"Inference {image_size}px"):
            images = images.to(DEVICE)
            with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == "cuda"):
                logits = model(images)
            probabilities.append(torch.softmax(logits.float(), dim=1).cpu().numpy())
    return np.concatenate(probabilities, axis=0)

probs_224 = predict_probabilities(224)
probs_256 = predict_probabilities(256)

original_predictions = probs_224.argmax(axis=1)
tta_predictions = (0.80 * probs_224 + 0.20 * probs_256).argmax(axis=1)
print("Predictions:", len(tta_predictions))

## 5. Create Submission Files

In [ ]:
submission_original = build_submission(
    test_df, sample_submission, original_predictions,
    SUBMISSION_DIR / "submission_B0.csv", num_classes=len(class_names),)
submission_tta = build_submission(
    test_df, sample_submission, tta_predictions,
    SUBMISSION_DIR / "submission_B0_TTA.csv", num_classes=len(class_names),)

print("Original:", SUBMISSION_DIR / "submission_B0.csv")
print("TTA:", SUBMISSION_DIR / "submission_B0_TTA.csv")
print("Submission shape:", submission_tta.shape)
display(submission_tta.head())

| File | Metode |
|---|---|
| `submission_B0.csv` | Prediksi original menggunakan resolusi 224 piksel |
| `submission_B0_TTA.csv` | Prediksi TTA menggunakan kombinasi resolusi 224 dan 256 piksel dengan bobot 80:20 |

Kedua file mengikuti format `sample_submission.csv`, yaitu terdiri dari kolom `id` dan `label`. Urutan ID dipertahankan agar setiap prediksi sesuai dengan gambar test yang benar.

File `submission_B0_TTA.csv` dipilih sebagai submission final karena menghasilkan performa validation yang lebih baik dibandingkan prediksi original.